# 组合策略移植模板（A股日线 · 池式开仓 · 事件查表）

适用：本地 `local_portfolio_backtest.py`（开仓池/平仓池引擎）策略 → BigQuant aistudio。
结构 6 cells：定义 → 取数 → 预计算 → 回测 → 诊断导出。策略特定逻辑集中在 **TODO** 标记处，
其余骨架（取数/状态管理/拒单回池/排序/执行）为 2026-09-01 strategy_006 三轮对账收敛后的
**已验证写法**，无特殊理由不要改。

## 使用步骤

1. 复制本模板 → 填 4 处 TODO（Cell 1 参数 / Cell 3 事件算法 / Cell 4 离场判定 / 优先级）
2. aistudio 逐 cell 运行（本地无 dai/bigtrader 时自动只定义不执行）
3. 跑最后的诊断导出 cell → 下载 `cloud_export/` 整个目录
4. 本地跑验收（七层对账，标准工具）：
   `python scripts/parity_check.py --cloud-export outputs/bigquant/cloud_export --panel <本地面板> \
    --events <本地事件> --warehouse <duckdb> --cloud-trades <云端csv> --local-trades <trades_paired.csv>`

**验收标准**：L1~L6 偏差浮点级；平仓同步率（±3天）≥95%；同步笔收益差 <1pp（≈佣金口径）；
分叉笔方向不单边倒（全晚卖 = 指标口径偏松或兜底未传导到 pivot；全早卖 = 口径偏紧）。

## 数据流纪律（防"修复未传导"类 bug）

**所有派生数据（指标/事件/透视）必须从同一个 `_ensure_indicators` 处理后的 panel 派生**。
禁止原始 panel 与处理后 panel 并存——strategy_006 二轮复盘实锤：Wilder ATR 只在
build_signals 内部 copy 重算、build_pivots 仍拿 SQL 的 SMA 口径 → 吊灯长期不触发。

## 本地引擎参数 → 云端等价实现对照

| 本地（local_portfolio_backtest） | 语义 | 云端等价 |
|---|---|---|
| `--warmup 60` | **双重语义**：全市场回测前 60 交易日禁开仓 + 个股 ≥60 bar | `WARMUP_NO_ENTRY_TD`（暖期边界）+ `WARMUP_BARS` |
| `--cost-bps 15` | 双边各 15bps，记在换手日 | `PerOrder(buy_cost=15e-4, sell_cost=15e-4, min_cost=0)` |
| `--fixed-notional 100000` | 固定名义虚拟记账，**可透支不复利** | `order_target_value(sym, NOTIONAL)`——bigtrader 真实现金，亏损后买不满属真实约束 |
| 池超时/失效 | 5 自然日 / 信号日收盘回撤 10% | ③ 池维护（超时用自然日差、回撤用信号日 close_sig） |
| 未成交 | 次日开盘 NaN → 跳过、**池保留**次日再试 | ⑥ 下单**不出池**、① 成交确认才出池（拒单自动由池重试） |
| 平仓价 NaN | fallback t 日收盘价 | 持仓残留 → ② 每 bar 补卖直到成交（跌停连板会推迟） |
| `select_order` | signal_date 降序；同日并列=池插入序 | ⑥ 同日并列=代码序（差异极小，可接受） |

## 已知执行差异（真实现金账户固有，对账时不算 bug）

真实现金（满额率<100%）/ 涨跌停与 volume_limit 撮合约束 / 停牌日跳过判定（本地用最近有效 bar）/
退市自动平仓。其余规则逐条对齐。

⚠ 指标口径：`m_ta_atr` = `m_avg(TR)` = **SMA 口径非 Wilder**（TR 突变期差 3~16%）。
凡本地用 ewm 的指标一律 `*_FALLBACK_PY=True` 走 pandas 兜底重算（口径对照见 bq_dai.md 等价 SQL）。


In [ ]:
# %% ═══════════════ Cell 1：定义（导入 + 参数）══════════════
from __future__ import annotations

import numpy as np
import pandas as pd

INDEX_CODE = "000300.SH"            # 基准/成分指数
START_PRECOMP = "2015-01-01"        # 预计算暖期起点（MA200/带形成；早于回测起点 2 年为宜）
BACKTEST_START = "2017-01-01"
BACKTEST_END = "2026-08-21"
CAPITAL_BASE = 1_000_000.0
POSITION_NOTIONAL = 100_000.0       # 固定名义/仓（不复利；云端为真实现金，亏损后可能买不满）
MAX_POSITIONS = 10
COST_BPS = 15.0                     # 双边各 15bps（=本地 cost_bps 口径）

# ── 引擎语义参数（对照本地 local_portfolio_backtest，勿漏）──
WARMUP_BARS = 60                    # 个股暖期：上市 <60 bar 不入池（本地 len(hd)<warmup）
WARMUP_NO_ENTRY_TD = 60             # 全市场暖期：回测首日起 60 个交易日内禁开新仓
                                    # （本地 --warmup 60 双重语义之一，漏掉会多出暖期交易）
POOL_TIMEOUT_DAYS = 5               # 开仓池超时（自然日）
POOL_DD = 0.10                      # 开仓池信号失效回撤（自信号日收盘）
ATR_N = 14
ATR_MULT = 4.0                      # 吊灯倍数（策略参数）

# ── TODO：策略自有参数（带宽/过滤/优先级等）──
MAX_DEGREE = 5                      # 示例：事件过滤

# ⚠ 指标口径：BigQuant m_ta_atr = m_avg(TR) = SMA，非本地 Wilder ewm(alpha=1/14)
#   （TR 突变期差 3~16%，翻转贴线吊灯判定）→ 默认 True 用 pandas Wilder 重算
ATR_FALLBACK_PY = True

print("Cell 1 完成：参数就绪")

In [ ]:
# %% ═══════════════ Cell 2：数据获取（dai + SQL → DataFrame）══════════════
# ⚠ dai.query 查分区表必须带 filters（date/instrument 范围），SQL where 不能替代
# ⚠ cn_stock_bar1d 为后复权价（与本地后复权面板一致，实测浮点级相同）
# ⚠ 指数日线表 = cn_stock_index_bar1d（cn_index_bar1d 不存在）

PANEL_SQL = f"""
select b.date, b.instrument, b.open, b.high, b.low, b.close,
       avg(b.close) over (partition by b.instrument order by b.date
                           rows between 19 preceding and current row) as ma20,
       stddev_samp(b.close) over (partition by b.instrument order by b.date
                           rows between 19 preceding and current row) as sd20,
       m_ta_atr(b.high, b.low, b.close, {ATR_N}) as atr{ATR_N}
from cn_stock_bar1d b
join (select distinct member_code from cn_stock_index_component
      where instrument = '{INDEX_CODE}') c
  on b.instrument = c.member_code
"""

INDEX_SQL_TPL = """
select date, instrument, close,
       avg(close) over (partition by instrument order by date
                        rows between 199 preceding and current row) as ma200
from {tbl} where instrument = '{idx}'
"""

COMP_SQL = f"""
select date, member_code from cn_stock_index_component
where instrument = '{INDEX_CODE}'
"""


def load_cloud_data():
    """SQL 取三类数据 → (panel_df, index_df, comp_df)。窗口全部 backward，无未来。"""
    import dai
    rg = {"date": [START_PRECOMP, BACKTEST_END]}
    panel = dai.query(PANEL_SQL, filters=rg).df()
    panel["date"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values(["instrument", "date"]).reset_index(drop=True)
    ma, sd = panel["ma20"], panel["sd20"]
    panel["ub"] = ma + 1.0 * sd              # BB(20,1.0)（示例；策略自带可删）
    panel["lb"] = ma - 1.0 * sd

    index_df, used = None, None
    for tbl in ("cn_stock_index_bar1d", "cn_index_bar1d"):
        try:
            index_df = dai.query(INDEX_SQL_TPL.format(tbl=tbl, idx=INDEX_CODE), filters=rg).df()
            used = tbl
            break
        except Exception as e:                          # noqa: BLE001
            print(f"  指数表 {tbl} 不可用：{e}")
    if index_df is None:
        raise RuntimeError("指数日线表两个候选名都不可用")
    print(f"  指数日线用表：{used}")
    index_df["date"] = pd.to_datetime(index_df["date"])

    comp = dai.query(COMP_SQL, filters=rg).df()
    comp["date"] = pd.to_datetime(comp["date"])
    return panel, index_df, comp


try:
    import dai                            # noqa: F401
    panel_df, index_df, comp_df = load_cloud_data()
    print(f"Cell 2 完成：面板 {len(panel_df)} 行 / {panel_df['instrument'].nunique()} 只 / "
          f"{panel_df['date'].min().date()}~{panel_df['date'].max().date()}；"
          f"指数 {len(index_df)} 行；成分 {len(comp_df)} 行")
except ImportError:
    print("本地环境无 dai：跳过取数（本地 parity 测试直接调 Cell 3 函数）")

In [ ]:
# %% ═══════════════ Cell 3：预计算（指标兜底 → 事件表 → 运行时结构 → 透视）══════════════
# 数据流纪律：所有派生数据（指标/事件/透视）必须从【同一个】_ensure_indicators 处理后的
# panel 派生（见末尾 try 块执行顺序）。禁止原始 panel 与处理后 panel 并存。


def _ensure_indicators(panel: pd.DataFrame) -> pd.DataFrame:
    """指标兜底重算（口径=本地共享模块：rolling ddof=1 / Wilder ewm）。

    ⚠ 调用方必须用返回值替换原 panel_df——只在内部 copy 重算、透视仍拿 SQL 口径
      是 strategy_006 二轮复盘的实锤 bug（吊灯长期不触发）。
    """
    def _per_symbol(col_fn):
        out = np.full(len(panel), np.nan)
        pos = 0
        for _, g in panel.groupby("instrument", sort=False):
            out[pos:pos + len(g)] = col_fn(g)
            pos += len(g)
        return out

    if panel["ma20"].dropna().empty:
        panel["ma20"] = _per_symbol(lambda g: g["close"].rolling(20).mean().to_numpy())
    if panel["sd20"].dropna().empty:
        panel["sd20"] = _per_symbol(lambda g: g["close"].rolling(20).std().to_numpy())
    # m_ta_atr = SMA(TR) ≠ 本地 Wilder → FALLBACK=True 时无条件重算
    if ATR_FALLBACK_PY or panel[f"atr{ATR_N}"].dropna().empty:
        def _wilder_atr(g):
            h, l, c = (g[x].to_numpy(float) for x in ("high", "low", "close"))
            pc = np.concatenate([[np.nan], c[:-1]])
            tr = np.maximum(h - l, np.maximum(np.abs(h - pc), np.abs(l - pc)))
            if len(tr):
                tr[0] = h[0] - l[0]
            return pd.Series(tr).ewm(alpha=1.0 / ATR_N, adjust=False,
                                     min_periods=ATR_N).mean().to_numpy()
        panel[f"atr{ATR_N}"] = _per_symbol(_wilder_atr)
    return panel


def build_signals(panel: pd.DataFrame, max_degree: int = MAX_DEGREE):
    """TODO：策略事件算法（本地 shared 模块同源逐行移植）。

    接口约定：逐标的跑状态机 → 返回 (events_raw, signals)；
    signals 需含列 instrument/date/close_sig/line/degree/sig_high/bars_at_signal，
    且同日同标的取 degree 最大者去重 + 过滤规则（=本地 EVENT_MAP 口径）。
    无未来：t 日事件仅由 ≤t 的 bar 决定；bars_at_signal = 该标的截至 t 的 bar 数。
    """
    # —— 示例骨架（策略特定，替换为本地算法同源实现）——
    rows = []
    for sym, g in panel.groupby("instrument", sort=True):
        g = g.reset_index(drop=True)
        # for e in <本地状态机>.run(g...):
        #     rows.append({"instrument": sym, **e})
    events_raw = pd.DataFrame(rows)
    if len(events_raw) == 0:
        empty = pd.DataFrame(columns=["instrument", "date", "degree", "line"])
        return empty, empty.copy()
    signals = (events_raw.sort_values("degree", ascending=False)
               .drop_duplicates(["instrument", "date"], keep="first"))
    signals = signals[signals["degree"] <= max_degree]
    return events_raw, signals.sort_values(["instrument", "date"]).reset_index(drop=True)


def build_runtime(index_df: pd.DataFrame, comp_df: pd.DataFrame) -> dict:
    """年线闸门 + 成分 asof + 全市场暖期边界（全部 ≤t 数据）。"""
    idx = index_df.sort_values("date").copy()
    idx["date"] = pd.to_datetime(idx["date"])
    bull = (idx["close"] > idx["ma200"]).fillna(False)
    comp = comp_df.copy()
    comp["date"] = pd.to_datetime(comp["date"])
    membership = {d: set(grp["member_code"]) for d, grp in comp.groupby("date")}
    bt_td = [d for d in sorted(idx["date"].unique()) if d >= pd.Timestamp(BACKTEST_START)]
    no_entry_before = bt_td[WARMUP_NO_ENTRY_TD] if len(bt_td) > WARMUP_NO_ENTRY_TD else None
    return {"bull_days": set(idx.loc[bull, "date"]),
            "member_dates": sorted(membership),
            "membership": membership,
            "no_entry_before": no_entry_before}


def build_pivots(panel: pd.DataFrame) -> dict:
    """离场判定透视。⚠ 必须传入已 _ensure_indicators 处理的 panel。"""
    return {"close": panel.pivot_table(index="date", columns="instrument", values="close").sort_index(),
            "high": panel.pivot_table(index="date", columns="instrument", values="high").sort_index(),
            "atr": panel.pivot_table(index="date", columns="instrument",
                                     values=f"atr{ATR_N}").sort_index()}


try:
    panel_df                              # noqa: F821
    panel_df = _ensure_indicators(panel_df)     # ← 纪律：先处理 panel 本体
    events_raw, signals = build_signals(panel_df)
    RT = build_runtime(index_df, comp_df)
    PIV = build_pivots(panel_df)                # ← 再从同一 panel 建透视
    SIG_BY_DATE = {d: g.to_dict("records") for d, g in signals.groupby("date")}
    _chk = PIV["atr"].stack()
    print(f"Cell 3 完成：事件 {len(events_raw)}（过滤后 {len(signals)}）；"
          f"年线上交易日 {len(RT['bull_days'])}；暖期禁开仓至 "
          f"{RT['no_entry_before'].date() if RT['no_entry_before'] is not None else '无'}；"
          f"ATR 非空 {(_chk == _chk).sum()}/{len(_chk)}（Wilder 重算生效）")
except NameError:
    print("本地环境（无 Cell 2 数据）：仅定义预计算函数")

In [ ]:
# %% ═══════════════ Cell 4：回测（bigtrader）——六步骨架（已验证写法，判定条件见 TODO）══════════════


def run_backtest():
    from bigquant import bigtrader

    def initialize(context: bigtrader.IContext):
        from bigtrader.finance.commission import PerOrder
        context.set_commission(PerOrder(buy_cost=COST_BPS / 10000,
                                        sell_cost=COST_BPS / 10000, min_cost=0))
        d = context.data
        context.sig_by_date = d["sig_by_date"]
        context.close_piv, context.high_piv, context.atr_piv = d["close_piv"], d["high_piv"], d["atr_piv"]
        context.bull_days = d["bull_days"]
        context.no_entry_before = d["no_entry_before"]
        context.member_dates, context.membership = d["member_dates"], d["membership"]
        context.st = {}      # sym -> {status: pending|active, order_date, signal_date, line, degree, post_high}
        context.pool = {}    # sym -> {signal_date, close_sig, line, degree, sig_high}
        print("[initialize] 事件日期 %d 个，年线上交易日 %d 个，暖期禁开仓至 %s"
              % (len(context.sig_by_date), len(context.bull_days),
                 context.no_entry_before.date() if context.no_entry_before is not None else "无"))

    def before_trading(context, data):
        pass

    def _px(piv, d, sym):
        try:
            v = piv.at[d, sym]
        except KeyError:
            return None
        return None if pd.isna(v) else float(v)

    def _members_asof(context, d):
        import bisect
        md = context.member_dates
        if d in context.membership:
            return context.membership[d]
        i = bisect.bisect_right(md, d)
        return context.membership[md[i - 1]] if i > 0 else set()

    def handle_data(context: bigtrader.IContext, data):
        d = pd.Timestamp(data.current_dt.strftime("%Y-%m-%d"))
        st, pool = context.st, context.pool
        positions = context.get_account_positions()

        # ① 昨日 pending 了结：已成交→active 且出池；未成交（停牌/一字板/资金不足拒单）
        #   → 仅撤状态、信号留池，后续日 ⑥ 重新下单（对齐本地"未成交跳过、池保留"）
        for sym in [s for s, v in st.items() if v["status"] == "pending" and v["order_date"] < d]:
            if sym in positions:
                st[sym]["status"] = "active"
                pool.pop(sym, None)
            else:
                st.pop(sym, None)

        # ② 账户有持仓但无状态（退市残留/卖单未成交）→ 每 bar 补卖直到成交
        for sym in set(positions.keys()) - set(st.keys()):
            context.order_target_value(sym, 0)

        # ③ 开仓池维护：信号后 N 自然日超时 / 收盘自信号日回撤失效
        for sym in list(pool):
            info = pool[sym]
            if (d - info["signal_date"]).days > POOL_TIMEOUT_DAYS:
                pool.pop(sym, None)
                continue
            c = _px(context.close_piv, d, sym)
            if c is not None and info.get("close_sig", 0) > 0 \
                    and c / info["close_sig"] - 1.0 < -POOL_DD:
                pool.pop(sym, None)

        # ④ 当日事件入池：全市场暖期闸 + 年线闸门 + 当日成分 + 个股暖期 + 未持仓未在池
        warmup_ok = context.no_entry_before is None or d >= context.no_entry_before
        if warmup_ok and d in context.bull_days:
            members = _members_asof(context, d)
            for ev in context.sig_by_date.get(d, []):
                sym = ev["instrument"]
                if sym in st or sym in pool or sym in positions:
                    continue
                if sym not in members or ev.get("bars_at_signal", 999) < WARMUP_BARS:
                    continue
                # TODO：策略自有入池过滤（degree/事件属性等）
                pool[sym] = {"signal_date": d, "close_sig": ev.get("close_sig"),
                             "line": ev.get("line"), "degree": ev.get("degree"),
                             "sig_high": ev.get("sig_high")}

        # ⑤ 离场（t 收盘判定 → t+1 开盘成交）
        for sym, v in list(st.items()):
            if v["status"] != "active":
                continue
            h = _px(context.high_piv, d, sym)
            if h is not None:
                v["post_high"] = max(v["post_high"], h)        # 信号日起含最高 high
            c = _px(context.close_piv, d, sym)
            if c is None:
                continue                                       # 停牌：当日不判定
            atr = _px(context.atr_piv, d, sym)
            # TODO：策略自有离场判定（示例=止损线优先 + 吊灯）
            stop_hit = v.get("line") is not None and c < v["line"]
            chan_hit = atr is not None and atr > 0 and c < v["post_high"] - ATR_MULT * atr
            if stop_hit or chan_hit:
                context.order_target_value(sym, 0)
                st.pop(sym, None)                              # 槽位立即可让（成交在 t+1 开盘）

        # ⑥ 开仓（最近信号优先/同日按代码序；固定名义；t+1 开盘成交）
        #    下单时【不】出池——成交确认在 ①（拒单由池自动重试）
        ordered = sorted(pool.items(), key=lambda kv: kv[0])                          # 同日稳定序
        ordered = sorted(ordered, key=lambda kv: kv[1]["signal_date"], reverse=True)  # 最近优先
        # TODO：策略自有优先级（如按 degree 排序，替换上面两行）
        slots = MAX_POSITIONS - len(st)
        for sym, info in ordered:
            if slots <= 0:
                break
            if sym in st or sym in positions:
                continue
            context.order_target_value(sym, POSITION_NOTIONAL)
            st[sym] = {"status": "pending", "order_date": d,
                       "signal_date": info["signal_date"], "line": info.get("line"),
                       "degree": info.get("degree"),
                       "post_high": info.get("sig_high") or 0.0}
            slots -= 1

    return bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date=BACKTEST_START,
        end_date=BACKTEST_END,
        capital_base=CAPITAL_BASE,
        initialize=initialize,
        before_trading_start=before_trading,
        handle_data=handle_data,
        order_price_field_buy="open",          # t 收盘决策 → t+1 开盘撮合
        order_price_field_sell="open",
        benchmark=INDEX_CODE,
        data={"sig_by_date": SIG_BY_DATE,
              "close_piv": PIV["close"], "high_piv": PIV["high"], "atr_piv": PIV["atr"],
              "bull_days": RT["bull_days"],
              "member_dates": RT["member_dates"], "membership": RT["membership"],
              "no_entry_before": RT["no_entry_before"]},
    )


def _bigtrader_ready() -> bool:
    try:
        from bigquant import bigtrader       # noqa: F401
        return hasattr(bigtrader, "run")
    except ImportError:
        pass
    except Exception as e:                   # noqa: BLE001
        print(f"bigtrader 引擎加载失败（本地环境常见）：{type(e).__name__}")
    return False


if _bigtrader_ready():
    performance = run_backtest()
    performance.render()
else:
    print("本地环境：Cell 4 定义完成但未执行（粘贴到 aistudio 运行）")

In [ ]:
# %% ═══════════════ Cell 5：诊断导出（标准验收步骤——跑完回测必跑本 cell）══════════════
# 产物 cloud_export/ 下载后放本地，再跑 skill scripts/parity_check.py 七层对账。
import os

try:
    panel_df, signals, index_df, comp_df, RT          # noqa: F821
    OUT = "cloud_export"
    os.makedirs(OUT, exist_ok=True)

    # 1) 原始+指标抽样：每标的每月最后交易日（全股票覆盖）
    probe = panel_df.copy()
    probe["ym"] = probe["date"].dt.to_period("M").astype(str)
    sel = probe.sort_values("date").groupby(["instrument", "ym"], as_index=False).tail(1)
    (sel[["instrument", "date", "open", "high", "low", "close", "ma20", "sd20", "ub", "lb",
          f"atr{ATR_N}"]]
     .to_csv(f"{OUT}/panel_probe.csv", index=False, encoding="utf-8-sig"))

    # 2) 中间：事件表（过滤后）
    signals.to_csv(f"{OUT}/signals.csv", index=False, encoding="utf-8-sig")

    # 3) 中间：闸门 + 暖期
    with open(f"{OUT}/runtime.txt", "w", encoding="utf-8") as f:
        f.write(f"no_entry_before={RT['no_entry_before']}\n")
        for d_ in sorted(RT["bull_days"]):
            f.write(f"bull={d_.date()}\n")

    # 4) 原始：指数日线（含 ma200）
    index_df.to_csv(f"{OUT}/index.csv", index=False, encoding="utf-8-sig")

    # 5) 原始：成分每月末快照
    c = comp_df.copy()
    c["ym"] = c["date"].dt.to_period("M").astype(str)
    last_d = c.groupby("ym")["date"].max()
    c[c["date"].isin(last_d)][["date", "member_code"]].to_csv(
        f"{OUT}/component_monthly.csv", index=False, encoding="utf-8-sig")

    print(f"Cell 5 完成：{OUT}/ → panel_probe {len(sel)} 行 / signals {len(signals)} 条。"
          "下载整个目录后本地跑：python scripts/parity_check.py --cloud-export <目录> ...")
except NameError:
    print("本地环境（无 Cell 2~4 数据）：Cell 5 仅在 aistudio 执行")